# Match outcomes: empirical adaptive Poisson

This notebook ties together:

- **`match_outcome_simple.ipynb`**: independent goals $X_A, X_B$; outcome probabilities are sums over a finite goal grid (e.g. $\mathbb{P}(A\text{ wins}) = \sum_{i>j} \mathbb{P}(X_A=i)\mathbb{P}(X_B=j)$).
- **`num_goals_distribution.ipynb`**: read `data/super_league/scores.csv` (`opponent:gf-ga`), build per-team mean goals for / against and the league mean $\mu$ per team–match row.
- **`docs/poisson_match_rates.tex`**: plug-in rates (attack $\times$ opponent defense),
  $$
  \lambda_A = \bar{s}_A \cdot \frac{\bar{c}_B}{\mu}, \qquad
  \lambda_B = \bar{s}_B \cdot \frac{\bar{c}_A}{\mu}.
  $$

Set **`TEAM_I`** and **`TEAM_J`** below (indices from `teams.yaml`). Team **I** is treated as **A** (home in notation only; no home advantage here unless you add it later).

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root as kernel cwd (idda/)
DATA = Path("data/super_league")

# Match to score: team index I = "A", team index J = "B"
TEAM_I = 1
TEAM_J = 14

# Truncate goal grid at 0..K_MAX (inclusive); increase if tail mass is large
K_MAX = 15

In [2]:
def load_idx_to_name(yaml_path: Path) -> dict[int, str]:
    out: dict[int, str] = {}
    for raw in yaml_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        name, _, rest = line.partition(":")
        out[int(rest.strip())] = name.strip()
    return out


def parse_goals(cell: str) -> tuple[int, int] | None:
    m = re.fullmatch(r"\d+:(\d+)-(\d+)", str(cell).strip())
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def team_means_from_scores(
    scores_csv: Path,
    week_cols: list[str] | None = None,
) -> tuple[dict[int, float], dict[int, float], float]:
    """Per team: mean GF, mean GA from played cells; league mean GF per row mu.

    If ``week_cols`` is None, use every column starting with ``w`` (full season in CSV).
    """

    df = pd.read_csv(scores_csv)
    if week_cols is None:
        week_cols = [c for c in df.columns if c.startswith("w")]
    gf_by: dict[int, list[int]] = {}
    ga_by: dict[int, list[int]] = {}
    all_gf: list[int] = []
    for _, row in df.iterrows():
        idx = int(row["team_idx"])
        gf_by.setdefault(idx, [])
        ga_by.setdefault(idx, [])
        for c in week_cols:
            p = parse_goals(row[c])
            if p is None:
                continue
            gf, ga = p
            gf_by[idx].append(gf)
            ga_by[idx].append(ga)
            all_gf.append(gf)
    mean_gf = {i: float(np.mean(v)) for i, v in gf_by.items() if v}
    mean_ga = {i: float(np.mean(v)) for i, v in ga_by.items() if v}
    mu = float(np.mean(all_gf)) if all_gf else float("nan")
    return mean_gf, mean_ga, mu


idx_to_name = load_idx_to_name(DATA / "teams.yaml")
mean_gf, mean_ga, MU = team_means_from_scores(DATA / "scores.csv")
print(f"League mean GF per team-match row μ = {MU:.4f}")

League mean GF per team-match row μ = 1.2925


In [3]:
def adaptive_lambdas(
    i: int,
    j: int,
    mean_gf: dict[int, float],
    mean_ga: dict[int, float],
    mu: float,
) -> tuple[float, float]:
    """
    λ for team i scoring vs j, and team j scoring vs i (docs/poisson_match_rates.tex).
    λ_i = s̄_i * c̄_j / μ,  λ_j = s̄_j * c̄_i / μ
    """
    if mu <= 0 or not np.isfinite(mu):
        raise ValueError("mu must be positive")
    si, ci = mean_gf[i], mean_ga[i]
    sj, cj = mean_gf[j], mean_ga[j]
    lam_i = si * (cj / mu)
    lam_j = sj * (ci / mu)
    return lam_i, lam_j


def poisson_pmf_up_to(k_max: int, lam: float) -> np.ndarray:
    """P(X=k) for k=0..k_max, stable via logs."""
    k = np.arange(k_max + 1, dtype=float)
    log_k_fact = np.zeros(k_max + 1)
    log_k_fact[1:] = np.cumsum(np.log(np.arange(1, k_max + 1)))
    log_p = k * np.log(lam) - lam - log_k_fact
    return np.exp(log_p)


def match_outcome_probs(
    lam_a: float,
    lam_b: float,
    k_max: int,
) -> tuple[float, float, float, float]:
    """
    Independent Poisson A,B on {0..k_max}. Returns (P_A_wins, P_tie, P_B_wins, tail_mass).
    tail_mass = 1 - P(grid); should be small if k_max is large enough.
    """
    pa = poisson_pmf_up_to(k_max, lam_a)
    pb = poisson_pmf_up_to(k_max, lam_b)
    mass = float(pa.sum() * pb.sum())
    # joint on grid renormalized for outcomes (optional strict normalization)
    pa_n = pa / pa.sum()
    pb_n = pb / pb.sum()
    p_a = 0.0
    p_tie = 0.0
    p_b = 0.0
    for ia in range(k_max + 1):
        for ib in range(k_max + 1):
            p = float(pa_n[ia] * pb_n[ib])
            if ia > ib:
                p_a += p
            elif ia == ib:
                p_tie += p
            else:
                p_b += p
    tail = 1.0 - mass
    return p_a, p_tie, p_b, tail

In [4]:
i, j = TEAM_I, TEAM_J
name_i = idx_to_name[i]
name_j = idx_to_name[j]

lam_i, lam_j = adaptive_lambdas(i, j, mean_gf, mean_ga, MU)

p_win_i, p_tie, p_win_j, tail = match_outcome_probs(lam_i, lam_j, K_MAX)

print(f"Match: team {i} ({name_i}) vs team {j} ({name_j})")
print(f"  mean GF: {mean_gf[i]:.3f} vs {mean_gf[j]:.3f}")
print(f"  mean GA: {mean_ga[i]:.3f} vs {mean_ga[j]:.3f}")
print(f"  λ_{i} (goals {name_i} vs {name_j}) = {lam_i:.4f}")
print(f"  λ_{j} (goals {name_j} vs {name_i}) = {lam_j:.4f}")
print()
print(f"Outcome probabilities (independent Poisson, grid 0..{K_MAX}, row-normalized):")
print(f"  P({name_i} wins) = {p_win_i:.4f}")
print(f"  P(draw)        = {p_tie:.4f}")
print(f"  P({name_j} wins) = {p_win_j:.4f}")
print(f"  Sum check      = {p_win_i + p_tie + p_win_j:.6f}")
print(f"  Mass off grid (approx) ≈ {tail:.5f} — increase K_MAX if non-negligible")

Match: team 1 (Galatasaray) vs team 14 (Fenerbahce)
  mean GF: 2.385 vs 2.259
  mean GA: 0.692 vs 1.037
  λ_1 (goals Galatasaray vs Fenerbahce) = 1.9132
  λ_14 (goals Fenerbahce vs Galatasaray) = 1.2101

Outcome probabilities (independent Poisson, grid 0..15, row-normalized):
  P(Galatasaray wins) = 0.5378
  P(draw)        = 0.2225
  P(Fenerbahce wins) = 0.2397
  Sum check      = 1.000000
  Mass off grid (approx) ≈ 0.00000 — increase K_MAX if non-negligible


In [5]:
# Week 28 fixtures from scores.csv: cell is `opponent_idx:…` (scores may be `?`).
# One row per undirected fixture; A = lower team_idx, B = higher (labels only).

WEEK_COL = "w28"


def parse_opponent_idx(cell: str) -> int | None:
    m = re.match(r"^(\d+):", str(cell).strip())
    return int(m.group(1)) if m else None


df_fix = pd.read_csv(DATA / "scores.csv")
if WEEK_COL not in df_fix.columns:
    raise ValueError(f"Missing column {WEEK_COL!r} in scores CSV")

fixture_rows: list[dict] = []
seen_pairs: set[tuple[int, int]] = set()
for _, row in df_fix.iterrows():
    i = int(row["team_idx"])
    j = parse_opponent_idx(row[WEEK_COL])
    if j is None:
        continue
    key = tuple(sorted((i, j)))
    if key in seen_pairs:
        continue
    seen_pairs.add(key)
    a, b = key
    lam_a, lam_b = adaptive_lambdas(a, b, mean_gf, mean_ga, MU)
    p_a, p_tie, p_b, _tail = match_outcome_probs(lam_a, lam_b, K_MAX)
    fixture_rows.append(
        {
            "idx_A": a,
            "idx_B": b,
            "team_A": idx_to_name[a],
            "team_B": idx_to_name[b],
            "λ_A": round(lam_a, 4),
            "λ_B": round(lam_b, 4),
            "P(A wins)": round(p_a, 4),
            "P(draw)": round(p_tie, 4),
            "P(B wins)": round(p_b, 4),
        }
    )

table_w28 = pd.DataFrame(fixture_rows).sort_values(["idx_A", "idx_B"]).reset_index(drop=True)
table_w28["sum"] = (
    table_w28["P(A wins)"] + table_w28["P(draw)"] + table_w28["P(B wins)"]
).round(6)
print(f"{WEEK_COL}: {len(table_w28)} fixtures (expected 9 for 18 teams)\n")
table_w28

w28: 9 fixtures (expected 9 for 18 teams)



,idx_A,idx_B,team_A,team_B,λ_A,λ_B,P(A wins),P(draw),P(B wins),sum
0,0,15,Gazisehir Gaziantep,Alanyaspor,1.2226,1.6110,0.2900,0.2458,0.4642,1.0
1,1,10,Galatasaray,Trabzonspor,1.9816,1.0514,0.5896,0.2146,0.1958,1.0
2,2,9,Samsunspor,Konyaspor,1.2465,1.0591,0.4053,0.2817,0.3130,1.0
3,3,7,Genclerbirligi,Goztepe,0.6172,1.2233,0.1873,0.3032,0.5095,1.0
4,4,8,Antalyaspor,Eyupspor,1.0082,0.8671,0.3781,0.3196,0.3023,1.0
5,5,16,Kasimpasa,Kayserispor,1.1717,0.8469,0.4337,0.3012,0.2651,1.0
6,6,12,Rizespor,Fatih Karagumruk,1.6576,0.9522,0.5394,0.2443,0.2163,1.0
7,11,13,Kocaelispor,Istanbul Basaksehir,0.7323,1.4943,0.1830,0.2625,0.5545,1.0
8,14,17,Fenerbahce,Besiktas,2.0069,1.4561,0.5043,0.2160,0.2797,1.0


In [6]:
# Backtest: train on weeks w1..w_TRAIN_THROUGH only, predict P(win/draw/loss) for w_PREDICT,
# then show the realized score (from the CSV). A/B order = lower vs higher team_idx (same as w28 table).

TRAIN_THROUGH_WEEK = 20
PREDICT_WEEK = 21

train_week_cols = [f"w{k}" for k in range(1, TRAIN_THROUGH_WEEK + 1)]
df_bt = pd.read_csv(DATA / "scores.csv")
missing_train = [c for c in train_week_cols if c not in df_bt.columns]
if missing_train:
    raise ValueError(f"Missing training columns: {missing_train}")

pred_col = f"w{PREDICT_WEEK}"
if pred_col not in df_bt.columns:
    raise ValueError(f"Missing prediction column {pred_col!r}")

# Aggregate here so this cell works even if an older kernel only has the 1-arg `team_means_from_scores`.
gf_by_bt: dict[int, list[int]] = {}
ga_by_bt: dict[int, list[int]] = {}
all_gf_bt: list[int] = []
for _, row in df_bt.iterrows():
    idx = int(row["team_idx"])
    gf_by_bt.setdefault(idx, [])
    ga_by_bt.setdefault(idx, [])
    for c in train_week_cols:
        p = parse_goals(row[c])
        if p is None:
            continue
        gf, ga = p
        gf_by_bt[idx].append(gf)
        ga_by_bt[idx].append(ga)
        all_gf_bt.append(gf)
mean_gf_bt = {i: float(np.mean(v)) for i, v in gf_by_bt.items() if v}
mean_ga_bt = {i: float(np.mean(v)) for i, v in ga_by_bt.items() if v}
MU_bt = float(np.mean(all_gf_bt)) if all_gf_bt else float("nan")

print(
    f"Train: {train_week_cols[0]}..{train_week_cols[-1]} → μ = {MU_bt:.4f} "
    f"(played cells only; `?` excluded)\nPredict: {pred_col}\n"
)

idx_row = df_bt.set_index("team_idx", drop=False)


def parse_opponent_idx(cell: str) -> int | None:
    m = re.match(r"^(\d+):", str(cell).strip())
    return int(m.group(1)) if m else None


backtest_rows: list[dict] = []
seen_pairs: set[tuple[int, int]] = set()
for _, row in df_bt.iterrows():
    i = int(row["team_idx"])
    j = parse_opponent_idx(row[pred_col])
    if j is None:
        continue
    key = tuple(sorted((i, j)))
    if key in seen_pairs:
        continue
    seen_pairs.add(key)
    a, b = key
    lam_a, lam_b = adaptive_lambdas(a, b, mean_gf_bt, mean_ga_bt, MU_bt)
    p_a, p_tie, p_b, _tail = match_outcome_probs(lam_a, lam_b, K_MAX)

    cell_a = idx_row.loc[a, pred_col]
    act = parse_goals(cell_a)
    if act is None:
        score_str = "— (unplayed or `?`)"
        outcome_actual = "—"
        max_prob_correct: bool | str = "—"
    else:
        g_a, g_b = act
        score_str = f"{g_a}–{g_b}"
        if g_a > g_b:
            outcome_actual = "A"
        elif g_a < g_b:
            outcome_actual = "B"
        else:
            outcome_actual = "draw"
        probs = {"A": p_a, "draw": p_tie, "B": p_b}
        pred_label = max(probs, key=probs.__getitem__)
        max_prob_correct = pred_label == outcome_actual

    backtest_rows.append(
        {
            "idx_A": a,
            "idx_B": b,
            "team_A": idx_to_name[a],
            "team_B": idx_to_name[b],
            "λ_A": round(lam_a, 4),
            "λ_B": round(lam_b, 4),
            "P(A wins)": round(p_a, 4),
            "P(draw)": round(p_tie, 4),
            "P(B wins)": round(p_b, 4),
            "actual A–B": score_str,
            "actual": outcome_actual,
            "argmax correct": max_prob_correct,
        }
    )

table_backtest = pd.DataFrame(backtest_rows).sort_values(["idx_A", "idx_B"]).reset_index(drop=True)
n_ok = sum(v is True for v in table_backtest["argmax correct"])
n_scored = int((table_backtest["actual"] != "—").sum())
if n_scored:
    print(f"Argmax hit rate (n={n_scored}): {n_ok}/{n_scored} = {n_ok / n_scored:.3f}\n")
table_backtest

Train: w1..w20 → μ = 1.3028 (played cells only; `?` excluded)
Predict: w21

Argmax hit rate (n=9): 6/9 = 0.667



,idx_A,idx_B,team_A,team_B,λ_A,λ_B,P(A wins),P(draw),P(B wins),actual A–B,actual,argmax correct
0,0,5,Gazisehir Gaziantep,Kasimpasa,1.3989,0.9787,0.4663,0.2705,0.2632,2–1,A,True
1,1,6,Galatasaray,Rizespor,2.7058,0.6448,0.8070,0.1287,0.0643,3–0,A,True
2,2,10,Samsunspor,Trabzonspor,1.0593,1.5313,0.2624,0.2558,0.4818,0–3,B,True
3,3,14,Genclerbirligi,Fenerbahce,0.8156,2.4179,0.1073,0.1650,0.7277,1–3,B,True
4,4,12,Antalyaspor,Fatih Karagumruk,1.4220,1.0765,0.4493,0.2652,0.2855,0–1,B,False
5,7,9,Goztepe,Konyaspor,1.7098,0.5527,0.6566,0.2283,0.1151,0–0,draw,False
6,8,13,Eyupspor,Istanbul Basaksehir,0.6448,1.8269,0.1249,0.2173,0.6578,1–2,B,True
7,11,16,Kocaelispor,Kayserispor,1.2588,0.6448,0.5126,0.2969,0.1906,2–1,A,True
8,15,17,Alanyaspor,Besiktas,0.9595,1.4776,0.2452,0.2628,0.4920,2–2,draw,False


In [17]:
# Walk-forward: train on w1..w_train_end, predict the next week, score argmax vs actual,
# then roll forward (train_end += 1) until we have predicted LAST_PREDICT_WEEK.
# Mean summary uses weeks with at least one scored fixture (nan if a week is all `?`).

FIRST_TRAIN_THROUGH_WEEK = 17
LAST_PREDICT_WEEK = 26

df_wf = pd.read_csv(DATA / "scores.csv")
idx_wf = df_wf.set_index("team_idx", drop=False)


def _parse_opponent_idx_wf(cell: str) -> int | None:
    m = re.match(r"^(\d+):", str(cell).strip())
    return int(m.group(1)) if m else None


def _aggregate_means_wf(df: pd.DataFrame, week_cols: list[str]) -> tuple[dict[int, float], dict[int, float], float]:
    gf_by: dict[int, list[int]] = {}
    ga_by: dict[int, list[int]] = {}
    all_gf: list[int] = []
    for _, row in df.iterrows():
        idx = int(row["team_idx"])
        gf_by.setdefault(idx, [])
        ga_by.setdefault(idx, [])
        for c in week_cols:
            p = parse_goals(row[c])
            if p is None:
                continue
            gf, ga = p
            gf_by[idx].append(gf)
            ga_by[idx].append(ga)
            all_gf.append(gf)
    mean_gf = {i: float(np.mean(v)) for i, v in gf_by.items() if v}
    mean_ga = {i: float(np.mean(v)) for i, v in ga_by.items() if v}
    mu = float(np.mean(all_gf)) if all_gf else float("nan")
    return mean_gf, mean_ga, mu


def _fixture_pairs_wf(df: pd.DataFrame, pred_col: str) -> list[tuple[int, int]]:
    seen: set[tuple[int, int]] = set()
    out: list[tuple[int, int]] = []
    for _, row in df.iterrows():
        i = int(row["team_idx"])
        j = _parse_opponent_idx_wf(row[pred_col])
        if j is None:
            continue
        key = tuple(sorted((i, j)))
        if key in seen:
            continue
        seen.add(key)
        out.append(key)
    out.sort()
    return out


walk_rows: list[dict] = []
for train_end in range(FIRST_TRAIN_THROUGH_WEEK, LAST_PREDICT_WEEK):
    predict_week = train_end + 1
    pred_col = f"w{predict_week}"
    if pred_col not in df_wf.columns:
        break
    train_cols = [f"w{k}" for k in range(1, train_end + 1)]
    mean_gf_wf, mean_ga_wf, MU_wf = _aggregate_means_wf(df_wf, train_cols)
    if not np.isfinite(MU_wf) or MU_wf <= 0:
        walk_rows.append(
            {
                "predict_week": predict_week,
                "train_through": train_end,
                "n_scored": 0,
                "argmax_hits": 0,
                "argmax_rate": float("nan"),
            }
        )
        continue

    hits = 0
    n_scored = 0
    for a, b in _fixture_pairs_wf(df_wf, pred_col):
        if a not in mean_gf_wf or b not in mean_gf_wf:
            continue
        lam_a, lam_b = adaptive_lambdas(a, b, mean_gf_wf, mean_ga_wf, MU_wf)
        p_a, p_tie, p_b, _ = match_outcome_probs(lam_a, lam_b, K_MAX)
        act = parse_goals(idx_wf.loc[a, pred_col])
        if act is None:
            continue
        g_a, g_b = act
        if g_a > g_b:
            actual = "A"
        elif g_a < g_b:
            actual = "B"
        else:
            actual = "draw"
        probs = {"A": p_a, "draw": p_tie, "B": p_b}
        pred_label = max(probs, key=probs.__getitem__)
        n_scored += 1
        if pred_label == actual:
            hits += 1
    rate = hits / n_scored if n_scored else float("nan")
    walk_rows.append(
        {
            "predict_week": predict_week,
            "train_through": train_end,
            "n_scored": n_scored,
            "argmax_hits": hits,
            "argmax_rate": round(rate, 4) if np.isfinite(rate) else rate,
        }
    )

table_walk = pd.DataFrame(walk_rows)
mean_over_weeks = float(np.nanmean(table_walk["argmax_rate"]))
print(
    f"Walk-forward: train first ends w{FIRST_TRAIN_THROUGH_WEEK}, "
    f"last predict w{LAST_PREDICT_WEEK}\n"
    f"Mean argmax hit rate over weeks (nanmean): {mean_over_weeks:.4f}\n"
)
table_walk

Walk-forward: train first ends w17, last predict w26
Mean argmax hit rate over weeks (nanmean): 0.5185



,predict_week,train_through,n_scored,argmax_hits,argmax_rate
0,18,17,9,5,0.5556
1,19,18,9,3,0.3333
2,20,19,9,6,0.6667
3,21,20,9,6,0.6667
4,22,21,9,4,0.4444
5,23,22,9,2,0.2222
6,24,23,9,5,0.5556
7,25,24,9,5,0.5556
8,26,25,9,6,0.6667
